## Pyomo 2: Problema del Transporte

In [1]:
from pyomo.environ import *
from pyomo.opt import SolverFactory, SolverStatus

PrintSolverOutput= False


### 2.1 Creamos los nodos y parámetros

In [2]:
modelo=ConcreteModel()

#Creamos los nodos de oferta y demanda
modelo.i=Set(initialize=['Cervecería A', 'Cervecería B'], doc='Cervecerías')
modelo.j=Set(initialize=['Bar 1','Bar 2', 'Bar 3', 'Bar 4', 'Bar 5'], doc='Bares')

In [3]:
#Definimos las capacidades de oferta y demanda, es decir los parámetros

modelo.a= Param(modelo.i, initialize={'Cervecería A':1000, 'Cervecería B':4000}, doc='Capacidades')

modelo.b= Param(modelo.j, initialize={'Bar 1':500, 'Bar 2':900, 'Bar 3':1800, 'Bar 4':200, 'Bar 5':700}, doc='Demanda de cada bar')

In [4]:
#Costo de transporte

costos={
    ('Cervecería A', 'Bar 1'):2,
    ('Cervecería A', 'Bar 2'):4,
    ('Cervecería A', 'Bar 3'):5,
    ('Cervecería A', 'Bar 4'):2,
    ('Cervecería A', 'Bar 5'):1,
    ('Cervecería B', 'Bar 1'):3,
    ('Cervecería B', 'Bar 2'):1,
    ('Cervecería B', 'Bar 3'):3,
    ('Cervecería B', 'Bar 4'):2,
    ('Cervecería B', 'Bar 5'):3,

}
modelo.d = Param(modelo.i, modelo.j, initialize=costos, doc='Costo de transporte')

### 2.2 Definimos la función de costo de transporte

In [5]:
def f_costo(modelo,i,j):
    return modelo.d[i,j]

#Definimos el costo de transporte
modelo.c=Param(modelo.i,modelo.j, initialize=f_costo, doc='Costo de transporte')

#Definimos variable x con las cantidades de cajas enviadas
modelo.x= Var(modelo.i,modelo.j,bounds=(0.0, None), doc='Capacidades')

$
x_{ij} \geq 0 \quad \forall \; i \in I, \; j \in J
$

Donde:

- $i$: cervecerías (fuentes de oferta).
- $j$: bares (destinos de demanda).
- $x_{ij}$: número de cajas transportadas desde la cervecería \(i\) hasta el bar \(j\).


### 2.3 Definimos las restricciones

In [6]:
# Restricción de oferta
def f_oferta(modelo, i):
    return sum(modelo.x[i, j] for j in modelo.j) <= modelo.a[i]

modelo.oferta = Constraint(
    modelo.i,
    rule=f_oferta,
    doc='Límites de oferta de cada cervecería'
)

# Restricción de demanda
def f_demanda(modelo, j):
    return sum(modelo.x[i, j] for i in modelo.i) == modelo.b[j]

modelo.demanda = Constraint(
    modelo.j,
    rule=f_demanda,
    doc='Límites de demanda de cada bar'
)

In [7]:
#Función objetivo
def f_objetivo(modelo):
    return sum(modelo.c[i, j] * modelo.x[i, j] for i in modelo.i for j in modelo.j)
modelo.objetivo = Objective(rule=f_objetivo, sense=minimize)

# Definir función de post-proceso
def pyomo_postprocess(options=None, instance=None, results=None):
    print("\n--- Resultados ---")
    print("Función Objetivo: ", value(instance.objetivo))
    instance.x.display()

# Usar GLPK en lugar de Gurobi
solver = SolverFactory("glpk", executable="/opt/homebrew/bin/glpsol")

# Resolver el modelo
results = solver.solve(modelo, tee=True)

# Imprimir resultados
print("\nSolución óptima encontrada\n" + '-'*80)
pyomo_postprocess(None, modelo, results)

GLPSOL--GLPK LP/MIP Solver 5.0
Parameter(s) specified in the command line:
 --write /var/folders/hb/c3zh8bns0mx4nw98_6k7rx8m0000gn/T/tmpylg7t4ui.glpk.raw
 --wglp /var/folders/hb/c3zh8bns0mx4nw98_6k7rx8m0000gn/T/tmpipy_69lo.glpk.glp
 --cpxlp /var/folders/hb/c3zh8bns0mx4nw98_6k7rx8m0000gn/T/tmpkg9iae8x.pyomo.lp
Reading problem data from '/var/folders/hb/c3zh8bns0mx4nw98_6k7rx8m0000gn/T/tmpkg9iae8x.pyomo.lp'...
7 rows, 10 columns, 20 non-zeros
70 lines were read
Writing problem data to '/var/folders/hb/c3zh8bns0mx4nw98_6k7rx8m0000gn/T/tmpipy_69lo.glpk.glp'...
57 lines were written
GLPK Simplex Optimizer 5.0
7 rows, 10 columns, 20 non-zeros
Preprocessing...
7 rows, 10 columns, 20 non-zeros
Scaling...
 A: min|aij| =  1.000e+00  max|aij| =  1.000e+00  ratio =  1.000e+00
Problem data seem to be well scaled
Constructing initial basis...
Size of triangular part is 7
      0: obj =   1.030000000e+04 inf =   1.000e+02 (1)
      1: obj =   1.020000000e+04 inf =   0.000e+00 (0)
*     4: obj =   8.6

### Modelo de optimización estocástica con un árbol de escenarios

In [8]:
#Crear modelo
model=ConcreteModel()

### Definimos los parámetros

In [9]:
T=[1,2]
PurchasePrice={
    1 :5.0,
    2 :5.0,
}
Demand={
    1:100,
    2:100
}
StorageInitial= 0.0
StorageCost= 1.0

In [10]:
# Variables de decisión
model.Purchase = Var(T, within=NonNegativeReals, doc='Purchase')
model.Storage = Var(T, within=NonNegativeReals, doc='Storage')
model.StorageInitial = Param(initialize=StorageInitial)

In [11]:
#Función objetivo
def Objetive(model):
    return sum(PurchasePrice[t]*model.Purchase[t] + StorageCost*model.Storage[t] for t in T)

model.TotalCost= Objective(rule=Objetive, sense=minimize)

In [12]:
#Restricciones
# Definir la restricción de balance de inventario
def Balance(model, t):
    if t == 1:
        return model.Storage[t] == model.StorageInitial + model.Purchase[t] - Demand[t]
    else:  # t >= 2
        return model.Storage[t] == model.Storage[t-1] + model.Purchase[t] - Demand[t]

# Anexar la restricción al modelo
model.StorageBalance = Constraint(T, rule=Balance)

# Función para mostrar resultados
def pyomo_postprocess(options=None, instance=None, results=None):
    print("Función Objetivo: " + str(value(instance.TotalCost)) + "\n")
    print("Compras por período:")
    instance.Purchase.display()
    print("\nInventario por período:")
    instance.Storage.display()

# Resolver el modelo usando Gurobi
opt = SolverFactory("glpk", executable="/opt/homebrew/bin/glpsol")
resultados = opt.solve(model)

# Post-proceso
pyomo_postprocess(None, model, resultados)

Función Objetivo: 1000.0

Compras por período:
Purchase : Purchase
    Size=2, Index={1, 2}
    Key : Lower : Value : Upper : Fixed : Stale : Domain
      1 :     0 : 100.0 :  None : False : False : NonNegativeReals
      2 :     0 : 100.0 :  None : False : False : NonNegativeReals

Inventario por período:
Storage : Storage
    Size=2, Index={1, 2}
    Key : Lower : Value : Upper : Fixed : Stale : Domain
      1 :     0 :   0.0 :  None : False : False : NonNegativeReals
      2 :     0 :   0.0 :  None : False : False : NonNegativeReals
